In [31]:
import pandas as pd
import numpy as np
import os

In [32]:
# File menggunakan pemisah ';' dan tidak memiliki header
df_stunting = pd.read_csv('../data/raw/dataset_stunting.csv', sep=';', header=None, skiprows=1)

In [33]:
# header yang sesuai dengan urutan kolom pada dataset
df_stunting.columns = [
    'id_anak', 'jenis_kelamin', 'usia_bulan', 'berat_badan', 'tinggi_badan',
    'status_bb_u', 'zscore_bb_u', 'status_tb_u', 'zscore_tb_u', 'status_bb_tb', 'zscore_bb_tb'
]

In [34]:
# Tampilkan informasi dasar tentang dataset
df_stunting.info()

<class 'pandas.DataFrame'>
RangeIndex: 40071 entries, 0 to 40070
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_anak        40071 non-null  int64  
 1   jenis_kelamin  40071 non-null  str    
 2   usia_bulan     40071 non-null  str    
 3   berat_badan    40071 non-null  float64
 4   tinggi_badan   40071 non-null  float64
 5   status_bb_u    40071 non-null  str    
 6   zscore_bb_u    40071 non-null  float64
 7   status_tb_u    40071 non-null  str    
 8   zscore_tb_u    40071 non-null  float64
 9   status_bb_tb   40066 non-null  str    
 10  zscore_bb_tb   40071 non-null  float64
dtypes: float64(5), int64(1), str(5)
memory usage: 3.4 MB


In [35]:
# konversi kolom usia_bulan ke tipe numerik
df_stunting['usia_bulan'] = pd.to_numeric(df_stunting['usia_bulan'], errors='coerce')

In [36]:
# Tampilkan informasi dasar tentang dataset setelah konversi
df_stunting.info()

<class 'pandas.DataFrame'>
RangeIndex: 40071 entries, 0 to 40070
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_anak        40071 non-null  int64  
 1   jenis_kelamin  40071 non-null  str    
 2   usia_bulan     40069 non-null  float64
 3   berat_badan    40071 non-null  float64
 4   tinggi_badan   40071 non-null  float64
 5   status_bb_u    40071 non-null  str    
 6   zscore_bb_u    40071 non-null  float64
 7   status_tb_u    40071 non-null  str    
 8   zscore_tb_u    40071 non-null  float64
 9   status_bb_tb   40066 non-null  str    
 10  zscore_bb_tb   40071 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 3.4 MB


In [37]:
# replace nilai '#N/A' dengan NaN untuk penanganan data yang hilang
df_stunting.replace('#N/A', np.nan, inplace=True)

# usia_bulan → isi median (numerik, 2 baris hilang)
df_stunting['usia_bulan'] = df_stunting['usia_bulan'].fillna(df_stunting['usia_bulan'].median())

# status_bb_tb → drop baris (kategorik, 5 baris hilang)
df_stunting.dropna(subset=['status_bb_tb'], inplace=True)
df_stunting.reset_index(drop=True, inplace=True)

In [38]:
# Mapping jenis_kelamin dari 'M' dan 'F' ke 'Laki-laki' dan 'Perempuan'
mapping_gender = {'M': 'Laki-laki', 'F': 'Perempuan'}
df_stunting['jenis_kelamin'] = df_stunting['jenis_kelamin'].map(mapping_gender)

In [39]:
# Mapping status gizi dan stunting ke bahasa Indonesia
mapping_status = {
    'Normal': 'Normal',
    'Stunted': 'Stunting',
    'Not Stunted': 'Tidak Stunting',
    'Malnutrition': 'Gizi Buruk',
    'Underfed': 'Gizi Kurang',
    'Thin': 'Kurus',
    'Very Thin': 'Sangat Kurus',
    'Obese': 'Obesitas',
    'Overnutrition': 'Gizi Lebih'
}

In [40]:
# Terapkan mapping status pada kolom status_bb_u, status_tb_u, dan status_bb_tb
kolom_status = ['status_bb_u', 'status_tb_u', 'status_bb_tb']
for col in kolom_status:
    df_stunting[col] = df_stunting[col].replace(mapping_status)

In [41]:
# Pastikan kolom usia_bulan sudah dalam tipe numerik
df_stunting['usia_bulan'] = df_stunting['usia_bulan'].astype(float)

In [42]:
# Tampilkan beberapa baris pertama untuk memastikan data sudah bersih dan terstruktur dengan baik
print(df_stunting[['id_anak', 'jenis_kelamin', 'usia_bulan', 'status_tb_u']].head())

   id_anak jenis_kelamin  usia_bulan status_tb_u
0        1     Perempuan        54.0    Stunting
1        2     Laki-laki        44.0    Stunting
2        3     Laki-laki        57.0    Stunting
3        4     Laki-laki        26.0    Stunting
4        5     Perempuan        59.0    Stunting


In [44]:
df_stunting.info()

<class 'pandas.DataFrame'>
RangeIndex: 40066 entries, 0 to 40065
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_anak        40066 non-null  int64  
 1   jenis_kelamin  40066 non-null  str    
 2   usia_bulan     40066 non-null  float64
 3   berat_badan    40066 non-null  float64
 4   tinggi_badan   40066 non-null  float64
 5   status_bb_u    40066 non-null  str    
 6   zscore_bb_u    40066 non-null  float64
 7   status_tb_u    40066 non-null  str    
 8   zscore_tb_u    40066 non-null  float64
 9   status_bb_tb   40066 non-null  str    
 10  zscore_bb_tb   40066 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 3.4 MB


In [43]:
# Simpan dataset yang sudah dibersihkan ke file baru
save_path = os.path.join('../data/processed', 'dataset_stunting_clean.csv')
df_stunting.to_csv(save_path, index=False)